# FraudeCero · Grupo 6 · 03 · Seguridad y privacidad (Ley 1581 de 2012)
**Proyecto Integrador Big Data – Uniciencia 2026-II** · Integrantes: Juan David Montoya · William Conde Villamizar


**Objetivo:** (1) clasificar las columnas por nivel de sensibilidad según la **Ley 1581 de 2012** y la **Ley 1266 de 2008** (dato financiero), (2) aplicar **2 técnicas de anonimización** mostrando el antes y el después —**hash SHA-256 con sal** y **enmascaramiento / generalización**— y (3) definir el **control de acceso por rol**. Al final se genera la **muestra anonimizada** (< 10 MB) que va en el paquete .zip.

## Configuración común
Esta celda es igual en todos los notebooks. Detecta si se ejecuta en **Google Colab** o en **Jupyter / GitHub Actions**, instala PySpark 3.5 + Delta Lake (OSS) y define:
- `LAKE`: carpeta del Lakehouse (capas `bronce`, `plata`, `oro`).
- `EVID`: carpeta `99_evidencias` (tablas de evidencia, log de monitoreo, tableros HTML).
- `guardar()` / `leer()`: escriben y leen tablas **Delta** (si Delta no está disponible usan Parquet y lo avisan).
- `registrar()`: escribe en el log de monitoreo el tiempo, las filas, el % de nulos y la memoria de cada etapa.

> En Colab cada notebook corre en una máquina distinta, por eso el Lakehouse se respalda en Google Drive al final de cada notebook y se restaura al inicio del siguiente.

In [ ]:
# ===================== CONFIGURACIÓN COMÚN FRAUDECERO =====================
NOTEBOOK = "03_seguridad"
import os, sys, time, json, shutil, datetime, subprocess, hashlib
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyspark==3.5.3", "delta-spark==3.2.1", "kagglehub", "psutil", "plotly"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DRIVE = "/content/drive/MyDrive/FraudeCero"
    LAKE = "/content/lakehouse"                      # disco local (rápido)
    LAKE_RESPALDO = f"{BASE_DRIVE}/lakehouse"        # respaldo persistente en Drive
    EVID = f"{BASE_DRIVE}/99_evidencias"
else:
    RAIZ = os.environ.get("FRAUDECERO_RAIZ", os.path.abspath(".."))
    LAKE = os.environ.get("FRAUDECERO_LAKE", os.path.join(RAIZ, "lakehouse"))
    LAKE_RESPALDO = None
    EVID = os.environ.get("FRAUDECERO_EVID", os.path.join(RAIZ, "99_evidencias"))

for carpeta in [LAKE, EVID, f"{EVID}/monitoreo", f"{EVID}/tablas"]:
    os.makedirs(carpeta, exist_ok=True)

def restaurar_lake():
    """En Colab: trae el Lakehouse desde Drive si el disco local está vacío."""
    if LAKE_RESPALDO and os.path.exists(LAKE_RESPALDO) and not os.listdir(LAKE):
        shutil.copytree(LAKE_RESPALDO, LAKE, dirs_exist_ok=True)
        print("Lakehouse restaurado desde Drive")

def respaldar_lake():
    """En Colab: copia el Lakehouse a Drive (sin la zona landing de archivos crudos)."""
    if LAKE_RESPALDO:
        shutil.copytree(LAKE, LAKE_RESPALDO, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("landing"))
        print("Lakehouse respaldado en Drive:", LAKE_RESPALDO)

restaurar_lake()

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

FORMATO = "delta"
try:
    from delta import configure_spark_with_delta_pip
    builder = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
               .config("spark.driver.memory", "6g")
               .config("spark.sql.shuffle.partitions", "8")
               .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
               .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.range(1).write.format("delta").mode("overwrite").save(f"{LAKE}/_prueba_delta")
except Exception as e:
    print("⚠️ Delta Lake no disponible, se usará Parquet. Detalle:", str(e)[:300])
    FORMATO = "parquet"
    spark = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
             .config("spark.driver.memory", "6g")
             .config("spark.sql.shuffle.partitions", "8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def ruta(capa, tabla):
    return f"{LAKE}/{capa}/{tabla}"

def leer(capa, tabla):
    df = spark.read.format(FORMATO).load(ruta(capa, tabla))
    df.createOrReplaceTempView(f"{capa}_{tabla}")
    return df

def guardar(df, capa, tabla, particion=None):
    w = df.write.format(FORMATO).mode("overwrite").option("overwriteSchema", "true")
    if particion:
        w = w.partitionBy(particion)
    w.save(ruta(capa, tabla))
    return leer(capa, tabla)

# ---------- Monitoreo: tiempo, filas, % nulos y memoria por etapa ----------
import psutil, pandas as pd
LOG_MON = f"{EVID}/monitoreo/log_ejecuciones.csv"
ARCHIVO_RUN = f"{EVID}/monitoreo/run_actual.txt"
RUN_ID = os.environ.get("FRAUDECERO_RUN_ID")
if not RUN_ID:
    if NOTEBOOK == "00_ingesta" or not os.path.exists(ARCHIVO_RUN):
        RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")   # cada corrida del pipeline empieza en 00
        with open(ARCHIVO_RUN, "w") as f:
            f.write(RUN_ID)
    else:
        RUN_ID = open(ARCHIVO_RUN).read().strip()
ORIGEN = "github_actions" if os.environ.get("GITHUB_ACTIONS") else ("colab" if EN_COLAB else "jupyter")

def memoria_mb():
    py = psutil.Process(os.getpid()).memory_info().rss / 1024**2
    try:
        rt = spark.sparkContext._jvm.java.lang.Runtime.getRuntime()
        jvm = (rt.totalMemory() - rt.freeMemory()) / 1024**2
    except Exception:
        jvm = 0.0
    return round(py + jvm, 1)

def registrar(etapa, t0, df=None, filas=None):
    """Registra una etapa en el log de monitoreo. t0 = time.time() al inicio de la etapa."""
    duracion = round(time.time() - t0, 2)
    pct_nulos = None
    if df is not None:
        filas = df.count()
        cols = df.columns
        nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).collect()[0]
        pct_nulos = round(sum(v or 0 for v in nulos) / (filas * len(cols)) * 100, 4) if filas else 0.0
    fila = {"run_id": RUN_ID, "origen": ORIGEN, "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            "notebook": NOTEBOOK, "etapa": etapa, "duracion_s": duracion, "filas": filas,
            "pct_nulos": pct_nulos, "memoria_mb": memoria_mb(), "cpu_pct": psutil.cpu_percent(interval=0.2),
            "formato": FORMATO}
    pd.DataFrame([fila]).to_csv(LOG_MON, mode="a", index=False, header=not os.path.exists(LOG_MON))
    print(f"⏱️ {etapa}: {duracion} s · filas={filas} · nulos={pct_nulos}% · mem={fila['memoria_mb']} MB")

def mostrar(df, n=20):
    """display() de Databricks no existe en Jupyter/Colab: se muestra como tabla pandas."""
    from IPython.display import display
    display(df.limit(n).toPandas() if hasattr(df, "toPandas") else df)

print(f"Notebook: {NOTEBOOK} · Entorno: {ORIGEN} · Formato de tablas: {FORMATO.upper()} · Spark {spark.version}")
print("LAKE =", LAKE, "| EVID =", EVID)

## 1. Clasificación de columnas (A2.4)
Categorías de la Ley 1581 de 2012 y el Decreto 1377 de 2013: **público**, **semiprivado**, **privado** y **sensible** (art. 5). Los datos de comportamiento financiero se tratan como **semiprivados** (Ley 1266 de 2008, art. 3).

In [ ]:
clasificacion = pd.DataFrame([
    {"columna": "id_transaccion", "fuente": "F1", "categoria_ley_1581": "Privado",
     "motivo": "Identificador directo: permite rastrear la operación hasta el titular de la tarjeta.",
     "tratamiento": "Hash SHA-256 con sal secreta + enmascaramiento para visualización"},
    {"columna": "Time", "fuente": "F1", "categoria_ley_1581": "Semiprivado",
     "motivo": "Cuasi-identificador: la marca de tiempo exacta combinada con el monto puede reidentificar.",
     "tratamiento": "Generalización a hora del día y franja horaria"},
    {"columna": "Amount", "fuente": "F1", "categoria_ley_1581": "Semiprivado (dato financiero, Ley 1266)",
     "motivo": "Revela comportamiento de consumo del titular.",
     "tratamiento": "Generalización a rangos de monto en vistas de negocio"},
    {"columna": "V1–V28", "fuente": "F1", "categoria_ley_1581": "Privado (seudonimizado)",
     "motivo": "Derivados por PCA de datos personales de la transacción; no son directamente identificables.",
     "tratamiento": "Solo en zona restringida (rol científico de datos); redondeo a 4 decimales en la muestra"},
    {"columna": "Class", "fuente": "F1", "categoria_ley_1581": "Privado",
     "motivo": "Indica que el titular fue víctima de fraude.",
     "tratamiento": "Acceso restringido; en oro solo agregado"},
    {"columna": "entidad", "fuente": "F2", "categoria_ley_1581": "Público",
     "motivo": "Nombre de la entidad vigilada publicado por la SFC.",
     "tratamiento": "Sin tratamiento"},
    {"columna": "valor / persona_natural / persona_juridica", "fuente": "F2", "categoria_ley_1581": "Público",
     "motivo": "Cifras agregadas publicadas en datos abiertos; no contienen titulares.",
     "tratamiento": "Sin tratamiento"},
])
clasificacion.to_csv(f"{EVID}/tablas/clasificacion_sensibilidad.csv", index=False)
mostrar(clasificacion)
print("📝 Ninguna columna es dato SENSIBLE del art. 5 (salud, origen racial, orientación política, biometría, etc.).")

## 2. Técnica 1 · Hash SHA-256 con sal
- El `id_transaccion` se reemplaza por `sha2(id || sal, 256)`.
- La **sal** es una clave secreta que **no se sube al repositorio** (`.gitignore`): sin ella no se puede reconstruir el id con un ataque de diccionario.
- El mismo id siempre produce el mismo hash, así que las tablas se pueden seguir uniendo.

In [ ]:
import secrets
t0 = time.time()
ruta_sal = f"{LAKE}/_secretos/sal.txt"
os.makedirs(os.path.dirname(ruta_sal), exist_ok=True)
SAL = os.environ.get("FRAUDECERO_SAL")
if not SAL:
    if not os.path.exists(ruta_sal):
        with open(ruta_sal, "w") as f:
            f.write(secrets.token_hex(32))
    SAL = open(ruta_sal).read().strip()

plata_cc = leer("plata", "transacciones") \
    .withColumn("id_transaccion", F.format_string("TX-%09d", F.col("id")))

con_hash = plata_cc.withColumn("id_hash", F.sha2(F.concat(F.col("id_transaccion"), F.lit(SAL)), 256))
print("ANTES → DESPUÉS (hash SHA-256)")
mostrar(con_hash.select("id_transaccion", "id_hash").limit(5))
registrar("seg_hash", t0)

## 3. Técnica 2 · Enmascaramiento y generalización
- **Enmascaramiento:** el id solo muestra los últimos 4 dígitos (`TX-*****0123`), útil para atención al cliente.
- **Generalización:** `Time` → hora y franja (madrugada, mañana, tarde, noche); `Amount` → rango de monto.

In [ ]:
t0 = time.time()
anon = (con_hash
        .withColumn("id_enmascarado", F.concat(F.lit("TX-*****"), F.substring("id_transaccion", -4, 4)))
        .withColumn("franja_horaria", F.when(F.col("hora") < 6, "madrugada").when(F.col("hora") < 12, "mañana")
                                       .when(F.col("hora") < 18, "tarde").otherwise("noche")))
print("ANTES (dato original)")
mostrar(anon.select("id_transaccion", "Time", "Amount").limit(5))
print("DESPUÉS (enmascarado y generalizado)")
mostrar(anon.select("id_enmascarado", "hora", "franja_horaria", "rango_monto").limit(5))
registrar("seg_enmascaramiento", t0)

## 4. Tablas protegidas y vistas por rol
| Rol | Puede ver | No puede ver |
|---|---|---|
| `ingeniero_datos` | bronce, plata, oro (todo) | — |
| `cientifico_datos` | `plata.transacciones_anonimizada` (hash, V1–V28, hora, rango, Class) | id original, Time exacto, Amount exacto |
| `analista_negocio` | `vw_negocio` (id enmascarado, franja, rango, Class) y capa oro | id, V1–V28, montos exactos |
| `auditor` | log de monitoreo, registro de fuentes y bitácoras | datos de transacciones |

In [ ]:
t0 = time.time()
cols_cientifico = ["id_hash", "dia", "hora", "franja_horaria", "rango_monto", "log_amount"] + \
                  [f"V{i}" for i in range(1, 29)] + ["Class"]
tabla_anon = guardar(anon.select(*cols_cientifico), "plata", "transacciones_anonimizada")
anon.select("id_enmascarado", "franja_horaria", "rango_monto", "Class").createOrReplaceTempView("vw_negocio")

PERMISOS = {
    "ingeniero_datos": {"bronce_transacciones", "plata_transacciones", "plata_transacciones_anonimizada", "vw_negocio",
                        "oro_kpis", "oro_fraude_por_hora", "oro_exposicion_entidad"},
    "cientifico_datos": {"plata_transacciones_anonimizada", "oro_kpis", "oro_fraude_por_hora"},
    "analista_negocio": {"vw_negocio", "oro_kpis", "oro_fraude_por_hora", "oro_exposicion_entidad"},
    "auditor": set(),
}

def consultar(rol, vista, n=3):
    """Simula el control de acceso: solo deja consultar las vistas autorizadas para el rol."""
    if vista not in PERMISOS.get(rol, set()):
        print(f"⛔ ACCESO DENEGADO · rol={rol} · objeto={vista}")
        return None
    print(f"✅ Acceso permitido · rol={rol} · objeto={vista}")
    return spark.table(vista).limit(n).toPandas()

leer("bronce", "transacciones"); leer("plata", "transacciones_anonimizada")
for o in ["kpis", "fraude_por_hora", "exposicion_entidad"]:
    leer("oro", o)
print(consultar("analista_negocio", "vw_negocio"))
consultar("analista_negocio", "plata_transacciones")
consultar("cientifico_datos", "bronce_transacciones")
print(consultar("cientifico_datos", "plata_transacciones_anonimizada").iloc[:, :6])
registrar("seg_vistas_rol", t0, df=tabla_anon)

### Control de acceso en Databricks / Unity Catalog (script para el workspace)
En Colab no hay Unity Catalog; este es el script equivalente que se aplica en Databricks Free Edition sobre el esquema del grupo:

In [ ]:
grants_sql = """
-- Grupos (Settings → Identity and access → Groups)
-- ingeniero_datos, cientifico_datos, analista_negocio, auditor
GRANT USE CATALOG ON CATALOG workspace TO `ingeniero_datos`, `cientifico_datos`, `analista_negocio`, `auditor`;
GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA workspace.fraudecero_bronce TO `ingeniero_datos`;
GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA workspace.fraudecero_plata  TO `ingeniero_datos`;
GRANT USE SCHEMA, SELECT          ON SCHEMA workspace.fraudecero_oro    TO `ingeniero_datos`, `analista_negocio`, `cientifico_datos`;
GRANT USE SCHEMA ON SCHEMA workspace.fraudecero_plata TO `cientifico_datos`, `analista_negocio`;
GRANT SELECT ON TABLE workspace.fraudecero_plata.transacciones_anonimizada TO `cientifico_datos`;
GRANT SELECT ON VIEW  workspace.fraudecero_plata.vw_negocio TO `analista_negocio`;
REVOKE ALL PRIVILEGES ON SCHEMA workspace.fraudecero_bronce FROM `cientifico_datos`, `analista_negocio`;
"""
with open(f"{EVID}/tablas/control_acceso_unity_catalog.sql", "w", encoding="utf-8") as f:
    f.write(grants_sql)
print(grants_sql)

## 5. Muestra anonimizada para el paquete .zip (A3.8, máx. 10 MB)
Todos los fraudes + 15.000 transacciones normales al azar, solo con columnas anonimizadas.

In [ ]:
t0 = time.time()
muestra = tabla_anon.sampleBy("Class", {0: 15000 / tabla_anon.filter("Class = 0").count(), 1: 1.0}, seed=7)
pdf = muestra.toPandas().round(4)
ruta_muestra = f"{EVID}/muestra_anonimizada.csv"
pdf.to_csv(ruta_muestra, index=False)
mb = os.path.getsize(ruta_muestra) / 1024**2
print(f"Muestra: {len(pdf):,} filas · {mb:.2f} MB · fraudes: {int(pdf.Class.sum())}")
assert mb < 10, "La muestra supera 10 MB"
registrar("seg_muestra", t0, filas=len(pdf))

## Cierre
Respalda el Lakehouse (en Colab) y confirma la ejecución completa.

In [ ]:
respaldar_lake()
print("✅ Notebook", NOTEBOOK, "terminado · run_id:", RUN_ID)